<a href="https://colab.research.google.com/github/yutongzou07/emotion-classification-mental-health-nlp/blob/main/emotion-classification-mental-health-nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Setups**

In [ ]:
#for distilbert later
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_START_METHOD"] = "thread"
os.environ["WANDB_API_KEY"] = "disabled"

In [ ]:
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, re
from collections import Counter
import pandas as pd
import numpy as np

emotion_path = "/content/drive/MyDrive/combined_emotion.csv"      # file with emotion labels (fear,joy,sad,anger)
sentiment_path = "/content/drive/MyDrive/combined_sentiment_data.csv"

In [ ]:
import random, os, numpy as np, torch
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
emotion_df = pd.read_csv(emotion_path)
sentiment_df = pd.read_csv(sentiment_path)

emotion_df.columns = [c.lower().strip() for c in emotion_df.columns]
sentiment_df.columns = [c.lower().strip() for c in sentiment_df.columns]

# 3. Extract columns (they may be named sentence,text,comment,…)
emotion_texts = emotion_df[emotion_df.columns[0]].astype(str).tolist()
emotion_labels = emotion_df[emotion_df.columns[1]].astype(str).tolist()

sent_texts = sentiment_df[sentiment_df.columns[0]].astype(str).tolist()
sent_labels = sentiment_df[sentiment_df.columns[1]].astype(str).tolist()

print("=== EMOTION TEXT SAMPLE ===\n")
print("\n".join(emotion_texts[:3]))    # like reviews[:1000]

print("\n=== EMOTION LABEL SAMPLE ===\n")
print("\n".join(emotion_labels[:10]))  # like labels[:20]

print("\n=== SENTIMENT TEXT SAMPLE ===\n")
print("\n".join(sent_texts[:3]))

print("\n=== SENTIMENT LABEL SAMPLE ===\n")
print("\n".join(sent_labels[:10]))

In [ ]:
from string import punctuation

# EMOTION DATASET
emotion_reviews = "\n".join(emotion_texts)

emotion_reviews = emotion_reviews.lower()
emotion_all_text = ''.join([c for c in emotion_reviews if c not in punctuation])

emotion_split = emotion_all_text.split('\n')
emotion_all_text = ' '.join(emotion_split)
emotion_words = emotion_all_text.split()

print("Emotion total words:", len(emotion_words))
print("Emotion first 20 words:", emotion_words[:20])


#print(emotion_words)
###############################################################



In [ ]:
# SENTIMENT DATASET
sent_reviews = "\n".join(sent_texts)

sent_reviews = sent_reviews.lower()
sent_all_text = ''.join([c for c in sent_reviews if c not in punctuation])

sent_split = sent_all_text.split('\n')
sent_all_text = ' '.join(sent_split)
sent_words = sent_all_text.split()

print("Sentiment total words:", len(sent_words))
print("Sentiment first 20 words:", sent_words[:20])

In [ ]:
# 1. combine all words from both datasets (or only emotion if you want separate models)
all_words = emotion_words + sent_words

# 2. get unique vocabulary
vocab = list(set(all_words))

# 3. create word → int mapping, starting at 1 (padding = 0 is unused here)
word_to_int = {word: i for i, word in enumerate(vocab, start=1)}
print(word_to_int)
# reverse mapping if needed
#int_to_word = {i: word for word, i in word_to_int.items()}

# 4. encode emotion dataset
emotion_reviews_int = []
for review in emotion_split:
    emotion_reviews_int.append([word_to_int[word] for word in review.split()])

# 5. encode sentiment dataset
sent_reviews_int = []
for review in sent_split:
    sent_reviews_int.append([word_to_int[word] for word in review.split()])


In [ ]:
# stats about vocabulary
print('Unique words: ', len((word_to_int)))  #76000+
print()

# print tokens in first review
print('Tokenized emotion review: \n', emotion_reviews_int[:1])

print('Tokenized sent review: \n', sent_reviews_int[:1])

In [ ]:
encoded_sent_labels = []

for label in sent_labels:
    if label.lower() == "positive":
        encoded_sent_labels.append(1)
    else:
        encoded_sent_labels.append(0)

print(encoded_sent_labels[:20])

In [ ]:
# outlier review stats
emotion_lens = Counter([len(x) for x in emotion_reviews_int])
print("Zero-length reviews: {}".format(emotion_lens[0])) # review_lens[0] counts the number of reviews with zero length
print("Maximum review length: {}".format(max(emotion_lens)))

sent_lens = Counter([len(x) for x in sent_reviews_int])
print("Zero-length reviews: {}".format(sent_lens[0])) # review_lens[0] counts the number of reviews with zero length
print("Maximum review length: {}".format(max(sent_lens)))

In [ ]:
# ==== EMOTION DATASET: pad/truncate to seq_length = 178 ====

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader



# emotion_reviews_ints must already exist (list of lists of ints)

# ---- PADDING FUNCTION ----
def pad_left(sequences, seq_length):
    arr = np.zeros((len(sequences), seq_length), dtype=np.int64)
    for i, seq in enumerate(sequences):
        if len(seq) <= seq_length:
            arr[i, -len(seq):] = np.array(seq, dtype=np.int64)
        else:
            arr[i, :] = np.array(seq[:seq_length], dtype=np.int64)
    return arr

In [ ]:
# Test your implementation!

seq_length = 178   # <-- changed

arr = pad_left(emotion_reviews_int, seq_length=seq_length)

## test statements - do not change - ##
assert len(arr)==len(emotion_reviews_int), "Your features should have as many rows as reviews."
assert len(arr[0])==seq_length, "Each feature row should contain seq_length values."

# print first 10 values of the first 30 batches
print(arr[:30,:10])



In [ ]:
print("Example emotion split:", emotion_split[:10])
print("Example encoded:", emotion_reviews_int[:10])
print("Any empty reviews?:", any(len(x)==0 for x in emotion_reviews_int))

In [ ]:
# Test your implementation for sentiment!

seq_length = 70   # max sentiment review length

arr_sent = pad_left(sent_reviews_int, seq_length=seq_length)

# test statements - do not change
assert len(arr_sent)==len(sent_reviews_int), "Your features should have as many rows as reviews."
assert len(arr_sent[0])==seq_length, "Each feature row should contain seq_length values."

print(arr_sent[:30,:10])

In [ ]:
#print("Example emotion split:", sent_split[:10])
#print("Example encoded:", sent_reviews_int[:10])
#print("Any empty reviews?:", any(len(x)==0 for x in sent_reviews_int))

In [ ]:
# Encode emotion labels (multiclass)
# emotion_labels must already exist as strings

# Make everything lowercase & cleaned
clean_emotion_labels = [str(l).strip().lower() for l in emotion_labels]

# Build mapping: each unique emotion -> integer
unique_emotions = sorted(list(set(clean_emotion_labels)))
emotion_label_to_int = {lab: i for i, lab in enumerate(unique_emotions)}

# Convert all labels into integers
emotion_int_labels = np.array([emotion_label_to_int[l] for l in clean_emotion_labels])

print("Emotion classes:", unique_emotions)
print("Mapping:", emotion_label_to_int)
print("First 20 encoded labels:", emotion_int_labels[:20])

In [ ]:
split_frac = 0.8

#features = arr      (emotion padded features)
#emotion_int_labels  (emotion integer labels)

features = arr
labels = np.array(emotion_int_labels)

## split data into training, validation, and test data (features and labels)

split_idx = int(len(features) * split_frac)

train_x, remaining_x = features[:split_idx], features[split_idx:]
train_y, remaining_y = labels[:split_idx], labels[split_idx:]

test_idx = int(len(remaining_x) * 0.5)

val_x, test_x = remaining_x[:test_idx], remaining_x[test_idx:]
val_y, test_y = remaining_y[:test_idx], remaining_y[test_idx:]

## print out the shapes of your resultant feature data
print("\t\t\tFeature Shapes:")
print("Train set: \t\t{}".format(train_x.shape),
      "\nValidation set: \t{}".format(val_x.shape),
      "\nTest set: \t\t{}".format(test_x.shape))


In [ ]:
split_frac = 0.8

# features = arr_sent      (sentiment padded features)
# encoded_sent_labels      (sentiment integer labels: 1/0)

features = arr_sent
labels = np.array(encoded_sent_labels)

## split data into training, validation, and test data (features and labels)

split_idx = int(len(features) * split_frac)

sent_train_x, remaining_x = features[:split_idx], features[split_idx:]
sent_train_y, remaining_y = labels[:split_idx], labels[split_idx:]

test_idx = int(len(remaining_x) * 0.5)

sent_val_x, sent_test_x = remaining_x[:test_idx], remaining_x[test_idx:]
sent_val_y, sent_test_y = remaining_y[:test_idx], remaining_y[test_idx:]

## print out the shapes of your resultant feature data
print("\t\t\tSentiment Feature Shapes:")
print("Train set: \t\t{}".format(sent_train_x.shape),
      "\nValidation set: \t{}".format(sent_val_x.shape),
      "\nTest set: \t\t{}".format(sent_test_x.shape))


In [ ]:
# Emotion: create TensorDatasets and DataLoaders
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# create Tensor datasets
train_data = TensorDataset(torch.from_numpy(train_x).long(), torch.from_numpy(train_y).long())
valid_data = TensorDataset(torch.from_numpy(val_x).long(),  torch.from_numpy(val_y).long())
test_data  = TensorDataset(torch.from_numpy(test_x).long(), torch.from_numpy(test_y).long())

# dataloaders
batch_size = 64  # <-- keeping consistent with previous cells

train_loader = DataLoader(train_data, shuffle=True,  batch_size=batch_size, drop_last=True)
valid_loader = DataLoader(valid_data, shuffle=False, batch_size=batch_size, drop_last=False)
test_loader  = DataLoader(test_data,  shuffle=False, batch_size=batch_size, drop_last=False)

print("Emotion DataLoaders created.")
print("Train / Val / Test sizes:", len(train_data), len(valid_data), len(test_data))
print("Batch size:", batch_size)

In [ ]:
dataiter = iter(train_loader)
sample_x, sample_y = next(dataiter)

print('Sample input size: ', sample_x.size())   # (batch_size, seq_length)
print('Sample input: \n', sample_x)
print()
print('Sample label size: ', sample_y.size())   # (batch_size)
print('Sample label: \n', sample_y)


In [ ]:
# Sentiment: create TensorDatasets and DataLoaders
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# create Tensor datasets
sent_train_data = TensorDataset(torch.from_numpy(sent_train_x).long(), torch.from_numpy(sent_train_y).long())
sent_valid_data = TensorDataset(torch.from_numpy(sent_val_x).long(),   torch.from_numpy(sent_val_y).long())
sent_test_data  = TensorDataset(torch.from_numpy(sent_test_x).long(),  torch.from_numpy(sent_test_y).long())

# dataloaders
batch_size = 64   # <-- consistent with emotion

sent_train_loader = DataLoader(sent_train_data, shuffle=True,  batch_size=batch_size, drop_last=True)
sent_valid_loader = DataLoader(sent_valid_data, shuffle=False, batch_size=batch_size, drop_last=False)
sent_test_loader  = DataLoader(sent_test_data,  shuffle=False, batch_size=batch_size, drop_last=False)

print("Sentiment DataLoaders created.")
print("Train / Val / Test sizes:", len(sent_train_data), len(sent_valid_data), len(sent_test_data))
print("Batch size:", batch_size)

In [ ]:
dataiter = iter(sent_train_loader)
sample_x, sample_y = next(dataiter)

print('Sample input size: ', sample_x.size())
print('Sample input: \n', sample_x)
print()
print('Sample label size: ', sample_y.size())
print('Sample label: \n', sample_y)

In [ ]:
# First checking if GPU is available
train_on_gpu=torch.cuda.is_available()

if(train_on_gpu):
    print('Training on GPU.')
else:
    print('No GPU available, training on CPU.')

# **Sentiment Analysis**

In [ ]:
# SentimentRNN class definition: __init__, forward, init_hidden
import torch
import torch.nn as nn

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=200, hidden_dim=128, n_layers=2, drop_prob=0.5):
        """
        Initialize the SentimentRNN model.
        Args:
            vocab_size (int): size of the vocabulary (largest token index + 1). Reserve 0 for PAD.
            embedding_dim (int): embedding vector size.
            hidden_dim (int): hidden size for LSTM.
            n_layers (int): number of LSTM layers.
            drop_prob (float): dropout probability between LSTM and FC.
        """
        super(SentimentRNN, self).__init__()
        self.n_layers = n_layers
        self.hidden_dim = hidden_dim

        # Embedding layer (padding_idx=0 ensures PAD tokens are not trained in embeddings)
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)

        # LSTM layer: batch_first=True so input shape is (batch, seq_len)
        self.lstm = nn.LSTM(input_size=embedding_dim,
                            hidden_size=hidden_dim,
                            num_layers=n_layers,
                            batch_first=True,
                            dropout=drop_prob if n_layers > 1 else 0.0)

        # Dropout + fully connected layer
        self.dropout = nn.Dropout(drop_prob)
        self.fc = nn.Linear(hidden_dim, 1)   # single output for binary classification
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, hidden):
        """
        Forward pass through the network.
        Args:
            x (LongTensor): input token IDs, shape (batch, seq_len)
            hidden (tuple): (h0, c0) initial hidden and cell states
        Returns:
            out_probs (Tensor): shape (batch,) probabilities in [0,1]
            hidden (tuple): (hn, cn) final hidden states
        """
        embeds = self.embedding(x)                 # (batch, seq_len, embedding_dim)
        lstm_out, hidden = self.lstm(embeds, hidden)  # lstm_out: (batch, seq_len, hidden_dim)

        # Take the output of the last time step
        last_out = lstm_out[:, -1, :]              # (batch, hidden_dim)
        out = self.dropout(last_out)
        out = self.fc(out)                         # (batch, 1)
        out = self.sigmoid(out)                    # (batch, 1)
        out = out.squeeze(1)                       # (batch,)
        return out, hidden

    def init_hidden(self, batch_size, device):
        """
        Initialize hidden and cell states to zeros, on given device.
        Args:
            batch_size (int)
            device (torch.device or string)
        Returns:
            (h0, c0): each shape (n_layers, batch_size, hidden_dim)
        """
        # get a reference tensor from parameters to ensure dtype/device consistency
        weight = next(self.parameters())
        h0 = weight.new_zeros(self.n_layers, batch_size, self.hidden_dim).to(device)
        c0 = weight.new_zeros(self.n_layers, batch_size, self.hidden_dim).to(device)
        return (h0, c0)


In [ ]:
# ==== Instantiate the SentimentRNN model ====

import torch

# --------------------------
# 1. Hyperparameters
# --------------------------

# detect vocab size from your word_to_int mapping
assert 'word_to_int' in globals(), "word_to_int not found — create vocabulary first."
vocab_size = max(word_to_int.values()) + 1    # +1 because indices start at 1

output_size = 1          # binary sentiment classifier → single sigmoid output
embedding_dim = 200      # We can increase to 300 for better performance
hidden_dim = 256         # common sizes: 128, 256, 512
n_layers = 2             # usually 2–3 work best
drop_prob = 0.5

print("Vocab size:", vocab_size)
print("Output size:", output_size)
print("Embedding dim:", embedding_dim)
print("Hidden dim:", hidden_dim)
print("Number of LSTM layers:", n_layers)

# --------------------------
# 2. Device setup
# --------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

# --------------------------
# 3. Instantiate the model
# --------------------------
sentiment_model = SentimentRNN(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    n_layers=n_layers,
    drop_prob=drop_prob
).to(device)

print("\nModel instantiated successfully:")
print(sentiment_model)


In [ ]:
[x for x in globals().keys() if 'sent' in x.lower()]

In [ ]:
import torch.optim as optim


# -------------------------
# Config / hyperparams
# -------------------------
lr = 1e-3
epochs = 8
print_every = 200     # validate & print every N train steps (set to len(sent_train_loader) for once-per-epoch)
clip = 5.0

# -------------------------
# Sanity checks
# -------------------------
assert 'sentiment_model' in globals(), "sentiment_model not found. Instantiate the model first."
assert 'sent_train_loader' in globals(), "sent_train_loader not found. Create sentiment DataLoaders first."
assert 'sent_valid_loader' in globals(), "sent_valid_loader not found. Create sentiment DataLoaders first."

# -------------------------
# Device & model setup
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_on_gpu = torch.cuda.is_available()
print("Using device:", device)

sentiment_model.to(device)
sentiment_model.train()

# -------------------------
# Loss & optimizer
# -------------------------
criterion = nn.BCELoss()   # model returns sigmoid probabilities
optimizer = torch.optim.Adam(sentiment_model.parameters(), lr=lr)

# -------------------------
# Training loop (your style)
# -------------------------
counter = 0

for e in range(epochs):
    # It's fine to init hidden at epoch start, but we'll re-init per-batch if batch size differs
    # Use the train loader's batch_size as a guideline if available
    try:
        base_batch_size = sent_train_loader.batch_size
    except Exception:
        base_batch_size = 64

    h = sentiment_model.init_hidden(base_batch_size, device)

    for inputs, labels in sent_train_loader:
        counter += 1

        # move to device and correct dtype
        inputs = inputs.to(device)
        labels = labels.float().to(device)    # BCELoss expects float

        # If the current batch size differs (last batch), re-init hidden to match
        if inputs.size(0) != h[0].size(1):   # h[0].shape == (n_layers, batch_size, hidden_dim)
            h = sentiment_model.init_hidden(inputs.size(0), device)

        # detach hidden to prevent backprop through entire history
        h = tuple([each.data for each in h])

        # 1) forward
        output, h = sentiment_model(inputs, h)   # output shape: (batch,)
        loss = criterion(output, labels)

        # 2) zero grad, backward, clip, step
        sentiment_model.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(sentiment_model.parameters(), clip)
        optimizer.step()

        # 3) optional: track training stats (small)
        if counter % print_every == 0:
            # Validation pass
            sentiment_model.eval()
            val_losses = []
            correct = 0
            total = 0

            with torch.no_grad():
                for val_inputs, val_labels in sent_valid_loader:
                    val_inputs = val_inputs.to(device)
                    val_labels = val_labels.float().to(device)
                    # init hidden for this val batch
                    val_h = sentiment_model.init_hidden(val_inputs.size(0), device)
                    val_output, val_h = sentiment_model(val_inputs, val_h)
                    val_loss = criterion(val_output, val_labels)
                    val_losses.append(val_loss.item())

                    preds = (val_output >= 0.5).long()
                    correct += (preds == val_labels.long()).sum().item()
                    total += val_labels.size(0)

            val_acc = 100.0 * correct / total if total > 0 else 0.0
            print("Epoch: {}/{} | Step: {} | Train Loss: {:.6f} | Val Loss: {:.6f} | Val Acc: {:.2f}%".format(
                e+1, epochs, counter, loss.item(), np.mean(val_losses) if val_losses else float('nan'), val_acc
            ))

            # back to train mode
            sentiment_model.train()

print("Training complete.")


In [ ]:
# ===== Clean Training Loop =====

import torch
import torch.nn as nn
import numpy as np

# -------------------
# Hyperparameters
# -------------------
lr = 0.001
epochs = 4
print_every = 100
clip = 5

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(sentiment_model.parameters(), lr=lr)

# -------------------
# Device
# -------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_on_gpu = torch.cuda.is_available()
sentiment_model.to(device)

print("Training on:", device)

# -------------------
# Training Loop
# -------------------
counter = 0

for e in range(epochs):
    sentiment_model.train()

    # Initialize hidden state for this epoch
    h = sentiment_model.init_hidden(batch_size, device)

    for inputs, labels in sent_train_loader:
        counter += 1

        # Move data to GPU if available
        inputs, labels = inputs.to(device), labels.float().to(device)

        # Handle last batch (possibly smaller size)
        if inputs.size(0) != h[0].size(1):
            h = sentiment_model.init_hidden(inputs.size(0), device)

        # Detach hidden state
        h = tuple([each.data for each in h])

        # ---- Forward pass ----
        output, h = sentiment_model(inputs, h)

        loss = criterion(output.squeeze(), labels)

        # ---- Backprop ----
        sentiment_model.zero_grad()
        loss.backward()

        # Clip exploding gradients
        nn.utils.clip_grad_norm_(sentiment_model.parameters(), clip)

        optimizer.step()

        # ---- PRINT & VALIDATE ----
        if counter % print_every == 0:
            sentiment_model.eval()

            val_losses = []
            correct, total = 0, 0

            with torch.no_grad():
                for val_inputs, val_labels in sent_valid_loader:
                    val_inputs = val_inputs.to(device)
                    val_labels = val_labels.float().to(device)

                    val_h = sentiment_model.init_hidden(val_inputs.size(0), device)
                    val_output, val_h = sentiment_model(val_inputs, val_h)

                    val_loss = criterion(val_output.squeeze(), val_labels)
                    val_losses.append(val_loss.item())

                    preds = (val_output.squeeze() >= 0.5).float()
                    correct += (preds == val_labels).sum().item()
                    total += val_labels.size(0)

            val_acc = 100 * correct / total

            print(f"Epoch: {e+1}/{epochs}",
                  f"Step: {counter}",
                  f"Loss: {loss.item():.6f}",
                  f"Val Loss: {np.mean(val_losses):.6f}",
                  f"Val Acc: {val_acc:.2f}%")

            sentiment_model.train()


In [ ]:
# === Sentiment model: Evaluated on the test set ===

import numpy as np
import torch
import torch.nn as nn

sentiment_model.eval()  # Switch to the evaluation mode

criterion = nn.BCELoss()   # Be consistent with the training
test_losses = []
correct = 0
total = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sentiment_model.to(device)

with torch.no_grad():
    for inputs, labels in sent_test_loader:
        inputs = inputs.to(device)
        labels = labels.float().to(device)

        # Initialize the hidden state for each batch
        h = sentiment_model.init_hidden(inputs.size(0), device)

        outputs, h = sentiment_model(inputs, h)

        loss = criterion(outputs.squeeze(), labels)
        test_losses.append(loss.item())

        # A probability greater than or equal to 0.5 is considered positive(1), otherwise negative(0).
        preds = (outputs.squeeze() >= 0.5).long()
        correct += (preds == labels.long()).sum().item()
        total += labels.size(0)

avg_test_loss = np.mean(test_losses)
test_acc = 100.0 * correct / total if total > 0 else 0.0

print(f"Sentiment Test Loss: {avg_test_loss:.4f}")
print(f"Sentiment Test Accuracy: {test_acc:.2f}%")


# **Emotions Analysis**

In [ ]:
# ==== Encode the emotion labels from strings to 0-5 ====

import numpy as np

# emotion_labels are the original labels (strings) read out from CSV.
unique_emotions = sorted(list(set(emotion_labels)))
print("Unique emotion labels:", unique_emotions)


label_to_int = {label: idx for idx, label in enumerate(unique_emotions)}
int_to_label = {idx: label for label, idx in label_to_int.items()}

print("Label -> Index:", label_to_int)

# Convert the string label of each sample to an integer
emotion_int_labels = np.array([label_to_int[l] for l in emotion_labels], dtype=np.int64)
print("First 10 encoded labels:", emotion_int_labels[:10])


In [ ]:
# ===== Emotion DataLoader =====
import torch
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64

# Convert to tensor
train_data = TensorDataset(torch.from_numpy(train_x).long(),
                           torch.from_numpy(train_y).long())
valid_data = TensorDataset(torch.from_numpy(val_x).long(),
                           torch.from_numpy(val_y).long())
test_data  = TensorDataset(torch.from_numpy(test_x).long(),
                           torch.from_numpy(test_y).long())

train_loader = DataLoader(train_data, shuffle=True,  batch_size=batch_size, drop_last=True)
valid_loader = DataLoader(valid_data, shuffle=False, batch_size=batch_size, drop_last=False)
test_loader  = DataLoader(test_data,  shuffle=False, batch_size=batch_size, drop_last=False)

print("Emotion train/val/test sizes:", len(train_data), len(valid_data), len(test_data))


In [ ]:
# ==== Define and instantiate the EmotionRNN model ====
import torch
import torch.nn as nn

class EmotionRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=200, hidden_dim=256,
                 n_layers=2, drop_prob=0.5, num_classes=6):
        super(EmotionRNN, self).__init__()

        self.n_layers = n_layers
        self.hidden_dim = hidden_dim

        # 1. embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # 2. LSTM
        self.lstm = nn.LSTM(embedding_dim,
                            hidden_dim,
                            num_layers=n_layers,
                            batch_first=True,
                            dropout=drop_prob)

        # 3. Dropout
        self.dropout = nn.Dropout(drop_prob)

        # 4. fully connected layer: hidden_dim -> num_classes
        self.fc = nn.Linear(hidden_dim, num_classes)
        # No explicit softmax is required; it comes with the CrossEntropyLoss

    def forward(self, x, hidden):
        # x: (batch_size, seq_len)
        embeds = self.embedding(x)                      # (batch, seq_len, embed_dim)
        lstm_out, hidden = self.lstm(embeds, hidden)    # (batch, seq_len, hidden_dim)

        # Only take the last time step
        out = lstm_out[:, -1, :]                        # (batch, hidden_dim)
        out = self.dropout(out)
        out = self.fc(out)                              # (batch, num_classes)

        return out, hidden

    def init_hidden(self, batch_size, device):
        weight = next(self.parameters()).data
        h0 = weight.new_zeros(self.n_layers, batch_size, self.hidden_dim).to(device)
        c0 = weight.new_zeros(self.n_layers, batch_size, self.hidden_dim).to(device)
        return (h0, c0)


# === Instantiate the model ===
vocab_size = max(word_to_int.values()) + 1
num_classes = len(unique_emotions)

embedding_dim = 200
hidden_dim = 256
n_layers = 2
drop_prob = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

emotion_model = EmotionRNN(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    n_layers=n_layers,
    drop_prob=drop_prob,
    num_classes=num_classes
).to(device)

print("Training device:", device)
print(emotion_model)




In [ ]:
# ==== Train EmotionRNN ====
import torch.optim as optim
import numpy as np

lr = 1e-3
epochs = 5
clip = 5.0
print_every = 200
batch_size = 64

criterion_emotion = nn.CrossEntropyLoss()
optimizer_emotion = optim.Adam(emotion_model.parameters(), lr=lr)

emotion_model.to(device)

print("Start training EmotionRNN...")

counter = 0

for e in range(epochs):
    emotion_model.train()
    h = emotion_model.init_hidden(batch_size, device)

    for inputs, labels in train_loader:
        counter += 1

        inputs = inputs.to(device)
        labels = labels.long().to(device)

        # If the size of the last batch is not equal to batch_size, hidden needs to be reinitialized
        if inputs.size(0) != h[0].size(1):
            h = emotion_model.init_hidden(inputs.size(0), device)

        # detach hidden to prevent the gradient from being passed back many steps earlier
        h = tuple([each.data for each in h])

        emotion_model.zero_grad()

        outputs, h = emotion_model(inputs, h)   # outputs: (batch, num_classes)
        loss = criterion_emotion(outputs, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(emotion_model.parameters(), clip)
        optimizer_emotion.step()

        # Regularly evaluate on the validation set
        if counter % print_every == 0:
            emotion_model.eval()
            val_losses = []
            correct = 0
            total = 0

            with torch.no_grad():
                for val_inputs, val_labels in valid_loader:
                    val_inputs = val_inputs.to(device)
                    val_labels = val_labels.long().to(device)

                    val_h = emotion_model.init_hidden(val_inputs.size(0), device)
                    val_outputs, val_h = emotion_model(val_inputs, val_h)

                    v_loss = criterion_emotion(val_outputs, val_labels)
                    val_losses.append(v_loss.item())

                    preds = torch.argmax(val_outputs, dim=1)
                    correct += (preds == val_labels).sum().item()
                    total += val_labels.size(0)

            val_acc = 100.0 * correct / total if total > 0 else 0.0

            print(f"Epoch: {e+1}/{epochs} | Step: {counter} | "
                  f"Train Loss: {loss.item():.4f} | "
                  f"Val Loss: {np.mean(val_losses):.4f} | "
                  f"Val Acc: {val_acc:.2f}%")

            emotion_model.train()

print("EmotionRNN training complete.")


In [ ]:
# save_checkpoint cell — run this after your training / after emotion_model is trained
import torch, os
SAVE_PATH = "./artifacts/best_emotion_model.pt"
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

ckpt = {
    "model_state_dict": emotion_model.state_dict(),
    "word_to_int": word_to_int,
    "label_to_int": label_to_int,
    "seq_length": train_x.shape[1]  # or seq_length variable if you kept it
}
torch.save(ckpt, SAVE_PATH)
print("Saved checkpoint to", SAVE_PATH)
# quick verify
loaded = torch.load(SAVE_PATH, map_location="cpu")
print("Checkpoint keys:", list(loaded.keys()))
print("Vocab size:", len(loaded["word_to_int"]))
print("Classes:", loaded["label_to_int"])
print("seq_length:", loaded["seq_length"])


In [ ]:
# ==== Make the final evaluation on the Emotion test set ====

emotion_model.eval()
test_losses = []
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.long().to(device)

        h = emotion_model.init_hidden(inputs.size(0), device)
        outputs, h = emotion_model(inputs, h)

        loss = criterion_emotion(outputs, labels)
        test_losses.append(loss.item())

        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

avg_test_loss = np.mean(test_losses)
test_acc = 100.0 * correct / total if total > 0 else 0.0

print(f"Emotion Test Loss: {avg_test_loss:.4f}")
print(f"Emotion Test Accuracy: {test_acc:.2f}%")
print("Index → Emotion:", int_to_label)


In [ ]:
# ==== Draw the Confusion Matrix ====
import torch
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# === Computational prediction and true labels ===
emotion_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.long().to(device)

        h = emotion_model.init_hidden(inputs.size(0), device)
        outputs, h = emotion_model(inputs, h)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# === confusion matrix ===
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=[int_to_label[i] for i in range(num_classes)],
            yticklabels=[int_to_label[i] for i in range(num_classes)])

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Emotion Classification Confusion Matrix (LSTM)")
plt.show()


In [ ]:
# ==== Single-sentence emotion prediction function ====
from string import punctuation

def encode_sentence_for_emotion(sentence, word_to_int, seq_length):
    # Clean up sentences
    sentence = sentence.lower()
    sentence = ''.join([c for c in sentence if c not in punctuation])
    words = sentence.split()

    # Encoding (unknown word = 0)
    encoded = [word_to_int.get(w, 0) for w in words]

    # padding(Right alignment)
    feature = np.zeros(seq_length)
    feature[-len(encoded):] = encoded[:seq_length]

    return torch.tensor(feature).long().unsqueeze(0)


def predict_emotion(sentence):
    emotion_model.eval()
    seq_length = train_x.shape[1]

    input_tensor = encode_sentence_for_emotion(sentence, word_to_int, seq_length).to(device)
    h = emotion_model.init_hidden(1, device)

    with torch.no_grad():
        output, h = emotion_model(input_tensor, h)
        pred = torch.argmax(output, dim=1).item()

    emotion = int_to_label[pred]
    print(f"Sentence: {sentence}")
    print(f"Predicted emotion: {emotion}")
    return emotion


In [ ]:
predict_emotion("I am so scared to go outside alone.")
predict_emotion("I feel extremely happy today!")
predict_emotion("I miss you and I love you so much.")
predict_emotion("This news makes me very sad.")
predict_emotion("Wow! I didn’t expect that at all!")

# **DISTILBERT training for emotion**

In [ ]:
!pip install transformers datasets evaluate -q

In [ ]:
#load emotion dataset
import pandas as pd

emotion_df = pd.read_csv("/content/drive/MyDrive/combined_emotion.csv")

# Reduce dataset to something manageable (e.g., 4000 samples)
emotion_df = emotion_df.sample(n=4000, random_state=42)


print(emotion_df.head())
print("Emotion classes:", emotion_df["emotion"].unique())

In [ ]:
#encode labels
from sklearn.preprocessing import LabelEncoder

emotion_le = LabelEncoder()
emotion_df["label"] = emotion_le.fit_transform(emotion_df["emotion"])

num_emotion_labels = len(emotion_le.classes_)
emotion_le.classes_, num_emotion_labels

In [ ]:
#data split
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(emotion_df, test_size=0.20, stratify=emotion_df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)

len(train_df), len(val_df), len(test_df)

In [ ]:
#convert to hugging face dataset
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df[["sentence", "label"]])
val_ds   = Dataset.from_pandas(val_df[["sentence", "label"]])
test_ds  = Dataset.from_pandas(test_df[["sentence", "label"]])

In [ ]:
#tokenize using distilbert
from transformers import DistilBertTokenizerFast

emotion_tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return emotion_tokenizer(batch["sentence"], padding="max_length", truncation=True, max_length=128)

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds = train_ds.remove_columns(["sentence"])
val_ds   = val_ds.remove_columns(["sentence"])
test_ds  = test_ds.remove_columns(["sentence"])

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

In [ ]:
#load distilbert
from transformers import DistilBertForSequenceClassification

emotion_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_emotion_labels
)

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    }

emotion_training_args = TrainingArguments(
    output_dir="./emotion_distilbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
)

emotion_trainer = Trainer(
    model=emotion_model,
    args=emotion_training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=emotion_tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
os.environ["WANDB_MODE"] = "offline"

In [ ]:
import wandb

# Initialize wandb in offline mode
wandb.init(mode="offline")

emotion_trainer.train()

# Moved wandb.finish() to after evaluation to ensure all logging is done

In [ ]:
emotion_trainer.evaluate(test_ds)
wandb.finish() # End the wandb run after evaluation

In [ ]:
#confusion matrix
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import os
import wandb # Ensure wandb is imported here

# Temporarily set WANDB_MODE to 'dryrun' (though re-initializing is the key fix here)
os.environ["WANDB_MODE"] = "dryrun"

# Re-initialize wandb for this prediction phase.
# Using reinit=True to ensure compatibility with older wandb versions.
wandb.init(mode="offline", reinit=True)

pred_output = emotion_trainer.predict(test_ds)
y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=list(emotion_le.classes_))
disp.plot(cmap="Blues", xticks_rotation=45)
plt.show()

# Finish the wandb run created for prediction
wandb.finish()

# Reset WANDB_MODE if needed for subsequent operations, or remove if no longer used.
# del os.environ["WANDB_MODE"] # Uncomment if you want to completely unset it

In [ ]:
# ==== Single-sentence prediction function for DistilBERT emotion model ====

import torch

def predict_emotion_distilbert(sentence):
    emotion_model.eval()

    # Tokenize the input
    inputs = emotion_tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=64
    )

    # Move tensors to GPU if available
    for k in inputs:
        inputs[k] = inputs[k].to(emotion_model.device)

    # Forward pass
    with torch.no_grad():
        outputs = emotion_model(**inputs)
        logits = outputs.logits
        pred_id = torch.argmax(logits, dim=1).item()

    # Convert back to label string
    pred_label = emotion_le.inverse_transform([pred_id])[0]

    print(f"Sentence: {sentence}")
    print(f"Predicted emotion (DistilBERT): {pred_label}\n")

    return pred_label

In [ ]:
predict_emotion_distilbert("I am so scared to go outside alone.")
predict_emotion_distilbert("I feel extremely happy today!")
predict_emotion_distilbert("I miss you and I love you so much.")
predict_emotion_distilbert("This news makes me very sad.")
predict_emotion_distilbert("Wow! I didn't expect that at all!")

# **DISTILBERT training for sentiment**

In [ ]:
#read sentiment dataset
sent_df = pd.read_csv("/content/drive/MyDrive/combined_sentiment_data.csv")
sent_df = sent_df.sample(n=3000, random_state=42) # Changed n from 4000 to 3000

print(sent_df.head())
print("Sentiment classes:", sent_df["sentiment"].unique())

In [ ]:
#encode sentiment labels
from sklearn.preprocessing import LabelEncoder

sent_le = LabelEncoder()
sent_df["label"] = sent_le.fit_transform(sent_df["sentiment"])

num_sent_labels = len(sent_le.classes_)
sent_le.classes_, num_sent_labels

In [ ]:
#data split
from sklearn.model_selection import train_test_split

sent_train_df, sent_temp_df = train_test_split(
    sent_df, test_size=0.20, stratify=sent_df["label"], random_state=42
)
sent_val_df, sent_test_df = train_test_split(
    sent_temp_df, test_size=0.50, stratify=sent_temp_df["label"], random_state=42
)

len(sent_train_df), len(sent_val_df), len(sent_test_df)

In [ ]:
#convert to hugging face dataset
sent_train_ds = Dataset.from_pandas(sent_train_df[["sentence", "label"]])
sent_val_ds   = Dataset.from_pandas(sent_val_df[["sentence", "label"]])
sent_test_ds  = Dataset.from_pandas(sent_test_df[["sentence", "label"]])

In [ ]:
#tokenize for distilbert
def tokenize_sent(batch):
    return emotion_tokenizer(batch["sentence"], padding="max_length",
                             truncation=True, max_length=128)

sent_train_ds = sent_train_ds.map(tokenize_sent, batched=True)
sent_val_ds   = sent_val_ds.map(tokenize_sent, batched=True)
sent_test_ds  = sent_test_ds.map(tokenize_sent, batched=True)

sent_train_ds = sent_train_ds.remove_columns(["sentence"])
sent_val_ds   = sent_val_ds.remove_columns(["sentence"])
sent_test_ds  = sent_test_ds.remove_columns(["sentence"])

sent_train_ds.set_format("torch")
sent_val_ds.set_format("torch")
sent_test_ds.set_format("torch")

In [ ]:
#load distilbert
from transformers import DistilBertForSequenceClassification

sent_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_sent_labels
)

In [ ]:
#training
sent_training_args = TrainingArguments(
    output_dir="./sentiment_distilbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    report_to="none" # Explicitly disable wandb reporting
)

In [ ]:
sent_trainer = Trainer(
    model=sent_model,
    args=sent_training_args,
    train_dataset=sent_train_ds,
    eval_dataset=sent_val_ds,
    tokenizer=emotion_tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
os.environ["WANDB_MODE"] = "offline"

In [ ]:
sent_trainer.train()

In [ ]:
sent_trainer.evaluate(sent_test_ds)

In [ ]:
#confusion matrix
sent_pred_output = sent_trainer.predict(sent_test_ds)
y_true_sent = sent_pred_output.label_ids
y_pred_sent = np.argmax(sent_pred_output.predictions, axis=1)

cm = confusion_matrix(y_true_sent, y_pred_sent)
disp = ConfusionMatrixDisplay(cm, display_labels=list(sent_le.classes_))
disp.plot(cmap="Blues")
plt.show()

In [ ]:
def predict_sentiment_distilbert(sentence):
    sent_model.eval()

    inputs = emotion_tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=64
    )

    for k in inputs:
        inputs[k] = inputs[k].to(sent_model.device)

    with torch.no_grad():
        outputs = sent_model(**inputs)
        logits = outputs.logits
        pred_id = torch.argmax(logits, dim=1).item()

    pred_label = sent_le.inverse_transform([pred_id])[0]

    print(f"Sentence: {sentence}")
    print(f"Predicted sentiment (DistilBERT): {pred_label}\n")

    return pred_label

In [ ]:
predict_sentiment_distilbert("This product is amazing, I love it!")
predict_sentiment_distilbert("This was a terrible experience.")

# **emotion-based response generation**

In [ ]:
!pip install -U transformers

In [1]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

In [ ]:
import os, re, time
import numpy as np
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
!pip install -U transformers accelerate bitsandbytes
!pip install -U protobuf

In [ ]:
import bitsandbytes as bnb
print(bnb.__version__)

In [ ]:
#configs
CHECKPOINT_PATH = "./artifacts/best_emotion_model.pt"
LLM_MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
USE_LLM_FOR_REPLY = True
MAX_LLM_TOKENS = 80
LLM_TEMPERATURE = 0.2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


print("DEVICE:", DEVICE)
print("CHECKPOINT:", CHECKPOINT_PATH)
print("LLM MODEL:", LLM_MODEL_ID, "| LLM ENABLED:", USE_LLM_FOR_REPLY)


In [ ]:
#small text utilities
from string import punctuation
def clean_text(s):
    s = str(s).strip().lower()
    s = ''.join([c for c in s if c not in punctuation])
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def encode_sentence_to_tensor(sentence, word_to_int, seq_length):
    s = clean_text(sentence)
    toks = s.split()
    arr = np.zeros(seq_length, dtype=np.int64)
    enc = [word_to_int.get(w, 0) for w in toks]
    if len(enc) == 0:
        return torch.from_numpy(arr).long().unsqueeze(0)
    if len(enc) <= seq_length:
        arr[-len(enc):] = enc
    else:
        arr[:] = enc[:seq_length]
    return torch.from_numpy(arr).long().unsqueeze(0)

In [ ]:
#model classes
class SelfAttentionPool(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    def forward(self, lstm_out, mask=None):
        scores = self.attn(lstm_out).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        pooled = torch.sum(weights * lstm_out, dim=1)
        return pooled, weights.squeeze(-1)

class EmotionBiLSTMAttn(nn.Module):
    def __init__(self, vocab_size, embedding_dim=200, hidden_dim=256, n_layers=2, dropout=0.5, num_classes=6, pretrained_embeddings=None, freeze_embeddings=False):
        super().__init__()
        assert hidden_dim % 2 == 0, "hidden_dim must be even"
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)
        if pretrained_embeddings is not None:
            if isinstance(pretrained_embeddings, np.ndarray):
                pretrained_embeddings = torch.tensor(pretrained_embeddings, dtype=torch.float32)
            self.embedding.weight.data.copy_(pretrained_embeddings)
            if freeze_embeddings:
                self.embedding.weight.requires_grad = False
        self.bilstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim // 2, num_layers=n_layers, batch_first=True, bidirectional=True, dropout=dropout if n_layers > 1 else 0.0)
        self.attn_pool = SelfAttentionPool(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        emb = self.embedding(x)
        lstm_out, _ = self.bilstm(emb)
        mask = (x != 0)
        pooled, attn_weights = self.attn_pool(lstm_out, mask=mask)
        out = self.dropout(pooled)
        logits = self.fc(out)
        return logits, attn_weights

In [ ]:
import re, torch, numpy as np
import torch.nn as nn

#define the original EmotionRNN used for training
class EmotionRNN_from_training(nn.Module):
    def __init__(self, vocab_size, embedding_dim=200, hidden_dim=256, n_layers=2, drop_prob=0.5, num_classes=6):
        super(EmotionRNN_from_training, self).__init__()
        self.n_layers = n_layers
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=drop_prob)
        self.dropout = nn.Dropout(drop_prob)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, hidden=None):
        # allow hidden to be optional for inference convenience
        embeds = self.embedding(x)                     # (B, T, E)
        if hidden is None:
            lstm_out, hidden = self.lstm(embeds)
        else:
            lstm_out, hidden = self.lstm(embeds, hidden)
        out = lstm_out[:, -1, :]                       # (B, hidden_dim)
        out = self.dropout(out)
        logits = self.fc(out)
        return logits, hidden

    def init_hidden(self, batch_size, device):
        weight = next(self.parameters()).data
        h0 = weight.new_zeros(self.n_layers, batch_size, self.hidden_dim).to(device)
        c0 = weight.new_zeros(self.n_layers, batch_size, self.hidden_dim).to(device)
        return (h0, c0)

#keep the BiLSTM+Attn class
class EmotionBiLSTMAttn_loader(nn.Module):
    def __init__(self, vocab_size, embedding_dim=200, hidden_dim=256, n_layers=2, drop_prob=0.5, num_classes=6, pretrained_embeddings=None, freeze_embeddings=False):
        super().__init__()
        assert hidden_dim % 2 == 0, "hidden_dim must be even"
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)
        if pretrained_embeddings is not None:
            if isinstance(pretrained_embeddings, np.ndarray):
                pretrained_embeddings = torch.tensor(pretrained_embeddings, dtype=torch.float32)
            self.embedding.weight.data.copy_(pretrained_embeddings)
            if freeze_embeddings:
                self.embedding.weight.requires_grad = False
        self.bilstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim // 2, num_layers=n_layers, batch_first=True, bidirectional=True, dropout=drop_prob if n_layers > 1 else 0.0)
        self.attn = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(drop_prob)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        lstm_out, _ = self.bilstm(emb)
        # simple mean/attn fallback (not used for loading the old model)
        mask = (x != 0)
        scores = self.attn(lstm_out).squeeze(-1)
        scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        pooled = torch.sum(weights * lstm_out, dim=1)
        out = self.dropout(pooled)
        logits = self.fc(out)
        return logits, weights.squeeze(-1)

#load checkpoint and inspect keys to decide which class to instantiate
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
sd_keys = list(ckpt["model_state_dict"].keys())
# quickly see prefix frequency
prefixes = sorted(list(set(k.split('.',1)[0] for k in sd_keys)))
print("state_dict top-level prefixes:", prefixes)

# Decide model type: if keys contain 'lstm.' then it's the original EmotionRNN_from_training
use_orig_emotionrnn = any(k.startswith("lstm.") or k.startswith("lstm_") or ".lstm." in k for k in sd_keys)
use_bilstm_attn = any(k.startswith("bilstm.") or k.startswith("attn_pool.") or ".bilstm." in k for k in sd_keys)

print("Detected original-LSTM keys:", use_orig_emotionrnn, "| Detected BiLSTM keys:", use_bilstm_attn)

# Build mapping objects from checkpoint
word_to_int = ckpt["word_to_int"]
label_to_int = ckpt["label_to_int"]
seq_length = int(ckpt["seq_length"])
if isinstance(list(label_to_int.keys())[0], str):
    int_to_label = {v:k for k,v in label_to_int.items()}
else:
    int_to_label = {int(k):v for k,v in label_to_int.items()}

vocab_size = max(word_to_int.values()) + 1
num_classes = len(int_to_label)

# Instantiate the matching class
if use_orig_emotionrnn and not use_bilstm_attn:
    print("Instantiating EmotionRNN_from_training (matches saved checkpoint)")
    loaded_model = EmotionRNN_from_training(vocab_size=vocab_size, embedding_dim=200, hidden_dim=256, n_layers=2, drop_prob=0.5, num_classes=num_classes)
    loaded_model.load_state_dict(ckpt["model_state_dict"])
elif use_bilstm_attn and not use_orig_emotionrnn:
    print("Instantiating EmotionBiLSTMAttn_loader (checkpoint looks like BiLSTM style)")
    loaded_model = EmotionBiLSTMAttn_loader(vocab_size=vocab_size, embedding_dim=200, hidden_dim=256, n_layers=2, drop_prob=0.5, num_classes=num_classes)
    loaded_model.load_state_dict(ckpt["model_state_dict"])
else:
    # ambiguous or mixed keys: try original LSTM first with strict=False to ignore mismatches
    try:
        print("Ambiguous keys — attempting to load with EmotionRNN_from_training (strict=False)")
        loaded_model = EmotionRNN_from_training(vocab_size=vocab_size, embedding_dim=200, hidden_dim=256, n_layers=2, drop_prob=0.5, num_classes=num_classes)
        missing, unexpected = loaded_model.load_state_dict(ckpt["model_state_dict"], strict=False)
        print("Loaded with strict=False. missing keys:", missing, "unexpected keys:", unexpected)
    except Exception as e:
        raise RuntimeError("Could not auto-rescue state_dict loading: " + str(e))

# Move to device, eval
loaded_model.to(DEVICE)
loaded_model.eval()
print("Loaded model on device:", DEVICE)

#small sanity forward pass
dummy = torch.zeros((1, seq_length), dtype=torch.long).to(DEVICE)
with torch.no_grad():
    # if model expects hidden arg, call accordingly
    try:
        out, _ = loaded_model(dummy)
    except TypeError:
        out = loaded_model(dummy)
print("Sanity forward shape:", out.shape)

# Save loaded_model into a name you expect downstream (so rest of notebook can use it)
emotion_model = loaded_model
print("emotion_model ready. Labels mapping:", int_to_label)


In [ ]:
#prediction helper
def predict_emotion(sentence, topk=3):
    x = encode_sentence_to_tensor(sentence, word_to_int, seq_length).to(DEVICE)
    with torch.no_grad():
        # EmotionRNN returns logits, hidden (tuple)
        logits, _ = emotion_model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pred_idx = int(np.argmax(probs))
    sorted_idx = list(np.argsort(-probs))
    top = [(int_to_label[i], float(probs[i])) for i in sorted_idx[:topk]]

    # return None for attention since EmotionRNN does not provide it
    return int_to_label[pred_idx], float(probs[pred_idx]), top, None

In [ ]:
#safety check
HIGH_RISK_KEYWORDS = {"suicide","kill myself","end my life","want to die","die by suicide","i cant go on","hurting myself","harm myself","hurt myself"}
ESCALATION_MESSAGE = "[ESCALATE] Possible self-harm or imminent danger detected. Contact local emergency services or escalate to human counselor."

def detect_high_risk(text):
    s = text.lower()
    for kw in HIGH_RISK_KEYWORDS:
        if kw in s:
            return True
    return False


In [ ]:
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM, pipeline

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

print("Loading Llama-3.2-1B in 8-bit mode...")

try:
    tokenizer_llm = AutoTokenizer.from_pretrained(LLM_MODEL_ID, trust_remote_code=True)
    model_llm = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    # ❗ Pipeline without device=
    llm_pipeline = pipeline(
        "text-generation",
        model=model_llm,
        tokenizer=tokenizer_llm
    )

    USE_LLM_FOR_REPLY = True
    print("Llama-3 (8-bit) loaded successfully!")

except Exception as e:
    print("ERROR loading Llama-3 (8-bit). Exception:")
    print(e)
    USE_LLM_FOR_REPLY = False
    llm_pipeline = None


In [ ]:
#LLM prompt & generator
def generate_reply_with_llm(user_text, emotion_label, max_new_tokens=MAX_LLM_TOKENS, temperature=LLM_TEMPERATURE):
    if llm_pipeline is None:
        raise RuntimeError("LLM pipeline is not available.")
    # concise system + conditioning
    prompt = (
        "You are a supportive assistant. You are NOT a therapist and must not give medical advice.\n"
        "If the user expresses self-harm or imminent danger, respond with exactly [ESCALATE] and emergency steps.\n"
        "Keep replies very short (1-3 sentences). Be empathetic, non-judgmental, and offer general coping ideas (breathing, grounding, contacting someone).\n\n"
        f"Emotion: {emotion_label}\nUser: {user_text}\nAssistant:"
    )
    out = llm_pipeline(prompt, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature, top_p=0.95, num_return_sequences=1)
    text = out[0].get("generated_text", out[0].get("text","")).strip()
    # remove prompt echo if present
    if text.startswith(prompt):
        text = text[len(prompt):].strip()
    return text


In [ ]:
#interactive loop
print("\nInteractive mode — type a sentence (or 'quit' to exit).")
while True:
    try:
        user_text = input("\nUser> ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nExiting interactive mode.")
        break
    if user_text.lower() in {"quit","exit","q"}:
        print("Bye.")
        break
    if len(user_text)==0:
        print("Please enter a sentence.")
        continue

    # pre-safety check
    if detect_high_risk(user_text):
        print("\n[HIGH-RISK] Detected possible self-harm. DO NOT auto-respond. Escalate to human.")
        print(ESCALATION_MESSAGE)
        continue

    # classify
    label, prob, topk, attn = predict_emotion(user_text, topk=3)
    print(f"\nPredicted emotion: {label}  (p={prob:.3f})")
    print("Top predictions:", topk)
    # show attention for last tokens
    toks = clean_text(user_text).split()
    if attn is None:
      print("(No attention weights — EmotionRNN does not use attention.)")
    else:
      toks = clean_text(user_text).split()
      attn_last = attn[-len(toks):].tolist()
      print("Token attention (right-aligned):", list(zip(toks, [float(x) for x in attn_last])))

    # LLM generation (only if loaded and allowed)
    if USE_LLM_FOR_REPLY:
        if llm_pipeline is None:
            print("LLM requested but not available. Check HF access / GPU memory.")
        else:
            # if classifier confidence is low, prefer a clarification (conservative)
            if prob < 0.55:
                print("\nLow confidence in emotion (p<0.55). Suggested clarification: ")
                print("\"I want to make sure I understand — could you tell me more about how you're feeling right now?\"")
            else:
                try:
                    reply = generate_reply_with_llm(user_text, label)
                    # post-filter LLM reply
                    if detect_high_risk(reply):
                        print("\n[LLM reply flagged as high-risk] — Escalate. Do not return automated reply.")
                        print(ESCALATION_MESSAGE)
                    else:
                        print("\nLLM reply:\n", reply)
                except Exception as e:
                    print("LLM generation error:", e)
    else:
        print("\nLLM generation disabled.")